%md

# 🛡️ Data Quality - Ecommerce Pedidos

## Squad 2 - Real Time for Business

### Grupo 3 - Ecommerce Pedidos

---

# 🎯 Objetivo

Este notebook tem como objetivo realizar a validação da qualidade
dos dados de pedidos antes da disponibilização para consumo
analítico.

Nesta etapa serão aplicadas regras técnicas e de negócio para
identificar possíveis inconsistências nos dados processados.

---

# 🏗️ Contexto da Arquitetura

O projeto utiliza arquitetura baseada em camadas:

```
RAW

↓

🥉 Bronze

↓

🛡️ Data Quality

↓

🥈 Silver

↓

🥇 Gold
```

A camada de Data Quality atua como uma etapa de controle,
garantindo que os dados estejam confiáveis antes das próximas
transformações.

---

# 📥 Origem dos Dados

Os dados analisados serão carregados da camada Bronze:

```
Database:

grupo3_bronze


Tabela:

ecommerce_pedidos
```

A Bronze contém:

- Dados ingeridos do Real Time;
- Estrutura original dos registros;
- Informações de auditoria;
- Identificação dos arquivos de origem.

---

# 🛡️ Regras de Qualidade Aplicadas

Nesta etapa serão realizadas validações técnicas e de negócio.

---

# 🔧 Validações Técnicas

## 1. Validação de Schema

Verificação da existência das colunas obrigatórias:

```
id_pedido

id_cliente

dt_pedido

status_pedido

valor_total

metodo_pagamento
```

Objetivo:

Garantir que novos arquivos recebidos mantenham a estrutura
esperada pelo pipeline.

---

## 2. Controle de Valores Financeiros

Validação do campo:

```
valor_total
```

Regra:

```
valor_total > 0
```

Objetivo:

Evitar valores negativos ou inconsistentes que possam afetar
indicadores financeiros.

---

## 3. Validação de Datas

Campos analisados:

```
dt_pedido

dt_previsao_entrega

dt_ultima_atualizacao_status
```

Objetivo:

Garantir que os campos temporais estejam em formato válido
e possam ser utilizados em análises.

---

## 4. Validação de Status

O campo:

```
status_pedido
```

deve possuir somente valores permitidos:

```
Entregue

Enviado

Em Separação

Cancelado

Pagamento Aprovado

Processando
```

Objetivo:

Evitar categorias inválidas no pipeline.

---

# 🔄 Controle de Processamento

Será implementado controle para evitar que o mesmo arquivo
Real Time seja processado mais de uma vez.

Objetivo:

- Evitar duplicidade;
- Garantir idempotência;
- Preservar indicadores corretos.

---

# 📊 Indicadores de Qualidade

Ao final das validações serão apresentados:

- Quantidade de registros analisados;
- Quantidade de registros inválidos;
- Regras aprovadas;
- Possíveis inconsistências encontradas.

---

# 🚦 Resultado das Validações

Cada regra será classificada como:

```
✅ Aprovada

ou

❌ Reprovada
```

Permitindo acompanhamento da qualidade dos dados.

---

# 🔄 Fluxo do Processo

```
grupo3_bronze.ecommerce_pedidos

        ↓

Regras de Data Quality

        ↓

Dados aprovados

        ↓

🥈 Silver / 🥇 Gold
```

---

# 🛡️ Boas Práticas Aplicadas

Durante esta etapa são aplicadas práticas de Engenharia
de Dados:

- Validação preventiva dos dados;
- Controle de qualidade;
- Monitoramento de inconsistências;
- Prevenção de duplicidades;
- Maior confiabilidade das informações.

---

# ✅ Resultado Esperado

Ao final deste notebook será possível:

- Garantir que os dados atendem aos critérios definidos;
- Identificar problemas antes do consumo analítico;
- Aumentar a confiabilidade do pipeline;
- Preparar os dados para futuras camadas de negócio.

Este notebook representa a etapa de qualidade dos dados
da Squad 2 - Grupo 3.

In [0]:
# ==================================================
# LEITURA DA CAMADA BRONZE
# DATA QUALITY
# ==================================================

df_quality = (
    spark
    .table(
        "grupo3_bronze.ecommerce_pedidos"
    )
)


print(
    "Dados Bronze carregados com sucesso!"
)


print(
    "Quantidade de registros:",
    df_quality.count()
)

Dados Bronze carregados com sucesso!
Quantidade de registros: 27


In [0]:
# ==================================================
# REGRA 1 - VALIDAÇÃO DE SCHEMA
# ==================================================

colunas_obrigatorias = [
    "id_pedido",
    "id_cliente",
    "dt_pedido",
    "status_pedido",
    "valor_total",
    "metodo_pagamento"
]

colunas_faltantes = [
    coluna
    for coluna in colunas_obrigatorias
    if coluna not in df_quality.columns
]


if not colunas_faltantes:
    print("Schema validado com sucesso!")
else:
    print(
        "Colunas faltantes:",
        colunas_faltantes
    )

Schema validado com sucesso!


In [0]:
# ==================================================
# IMPORTAÇÃO DAS BIBLIOTECAS
# DATA QUALITY
# ==================================================

from pyspark.sql.functions import (
    col,
    count,
    when
)

print(
    "Bibliotecas Data Quality carregadas com sucesso!"
)

Bibliotecas Data Quality carregadas com sucesso!


In [0]:
# ==================================================
# LEITURA DA CAMADA BRONZE
# DATA QUALITY
# ==================================================

df_quality = (
    spark
    .table(
        "grupo3_bronze.ecommerce_pedidos"
    )
)


print(
    "Dados Bronze carregados com sucesso!"
)


print(
    "Quantidade de registros:",
    df_quality.count()
)

Dados Bronze carregados com sucesso!
Quantidade de registros: 27


In [0]:
# ==================================================
# SCHEMA DOS DADOS
# ==================================================

df_quality.printSchema()

root
 |-- id_pedido: long (nullable = true)
 |-- id_cliente: long (nullable = true)
 |-- id_endereco_entrega: long (nullable = true)
 |-- dt_pedido: timestamp (nullable = true)
 |-- status_pedido: string (nullable = true)
 |-- valor_total: double (nullable = true)
 |-- valor_frete: double (nullable = true)
 |-- metodo_pagamento: string (nullable = true)
 |-- dt_previsao_entrega: timestamp (nullable = true)
 |-- dt_ultima_atualizacao_status: timestamp (nullable = true)
 |-- bronze_source_file: string (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)



In [0]:
# ==================================================
# REGRA 1 - VALIDAÇÃO DE SCHEMA
# ==================================================

colunas_obrigatorias = [
    "id_pedido",
    "id_cliente",
    "dt_pedido",
    "status_pedido",
    "valor_total",
    "metodo_pagamento"
]


colunas_faltantes = [
    coluna
    for coluna in colunas_obrigatorias
    if coluna not in df_quality.columns
]


if not colunas_faltantes:

    print(
        "Schema validado com sucesso!"
    )

else:

    print(
        "Colunas faltantes:",
        colunas_faltantes
    )

Schema validado com sucesso!


In [0]:
# ==================================================
# REGRA 2 - VALIDAÇÃO VALOR TOTAL
# ==================================================

valores_invalidos = (
    df_quality
    .filter(
        col("valor_total") <= 0
    )
)


quantidade_invalidos = (
    valores_invalidos
    .count()
)


if quantidade_invalidos == 0:

    print(
        "Validação de valor_total aprovada!"
    )

else:

    print(
        "Valores inválidos encontrados:",
        quantidade_invalidos
    )

Valores inválidos encontrados: 1


In [0]:
# ==================================================
# VISUALIZAÇÃO DOS VALORES INVÁLIDOS
# ==================================================

display(
    valores_invalidos
)

id_pedido,id_cliente,id_endereco_entrega,dt_pedido,status_pedido,valor_total,valor_frete,metodo_pagamento,dt_previsao_entrega,dt_ultima_atualizacao_status,bronze_source_file,bronze_ingested_at
1790001004,1790001001,1790001001,2026-09-21T11:29:29.135Z,Pagamento Aprovado,-1.92,13.7,pix,2026-09-27T11:29:29.135Z,2026-09-21T11:29:29.135Z,real-time-data/2026/09/21/113002/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z


## Resultado da Regra 2 - Valor Total

Foi identificado 1 registro inconsistente.

Regra:

valor_total deve ser maior que zero.

Resultado:

❌ Falha encontrada

Registro afetado:

id_pedido: 1790001004

valor_total: -1.92

Arquivo origem:

real-time-data/2026/09/21/113002/ecommerce_pedidos.parquet

In [0]:
# ==================================================
# REGRA 3 - VALIDAÇÃO DAS DATAS
# ==================================================

campos_data = [
    "dt_pedido",
    "dt_previsao_entrega",
    "dt_ultima_atualizacao_status"
]


for campo in campos_data:

    invalidos = (
        df_quality
        .filter(
            col(campo)
            .isNull()
        )
        .count()
    )

    if invalidos == 0:

        print(
            campo,
            ": Data válida"
        )

    else:

        print(
            campo,
            ":",
            invalidos,
            "valores inválidos"
        )

dt_pedido : Data válida
dt_previsao_entrega : Data válida
dt_ultima_atualizacao_status : Data válida


In [0]:
# ==================================================
# REGRA 4 - VALIDAÇÃO DE STATUS
# ==================================================

status_validos = [
    "Entregue",
    "Enviado",
    "Em Separação",
    "Cancelado",
    "Pagamento Aprovado",
    "Processando"
]


status_invalidos = (
    df_quality
    .filter(
        ~col("status_pedido")
        .isin(status_validos)
    )
)


quantidade_status_invalidos = (
    status_invalidos
    .count()
)


if quantidade_status_invalidos == 0:

    print(
        "Validação de status aprovada!"
    )

else:

    print(
        "Status inválidos encontrados:",
        quantidade_status_invalidos
    )

Status inválidos encontrados: 2


In [0]:
# ==================================================
# VISUALIZAÇÃO DOS STATUS INVÁLIDOS
# ==================================================

display(
    status_invalidos
)

id_pedido,id_cliente,id_endereco_entrega,dt_pedido,status_pedido,valor_total,valor_frete,metodo_pagamento,dt_previsao_entrega,dt_ultima_atualizacao_status,bronze_source_file,bronze_ingested_at
1790000559,1790000561,1790000562,2026-09-21T11:22:04.299Z,Aguardando,65.0,13.35,boleto,2026-09-29T11:22:04.299Z,2026-09-21T11:22:04.299Z,real-time-data/2026/09/21/112245/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z
1790001005,1790001000,1790001000,2026-09-21T11:29:51.135Z,Aguardando,1675.96,0.0,cartão débito,2026-09-26T11:29:51.135Z,2026-09-21T11:29:51.135Z,real-time-data/2026/09/21/113002/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z


## Resultado Regra 4 - Status Pedido

Foram identificados 2 registros com valores
fora do domínio permitido.

Regra:

status_pedido deve pertencer ao conjunto esperado.

Resultado:

❌ Falha encontrada

Amostras devem ser analisadas antes da
disponibilização para camadas analíticas.

## Resultado Regra 4 - Status Pedido

Foram encontrados 2 registros com status fora do
domínio inicialmente definido.

Status encontrado:

Aguardando

Registros afetados:

- id_pedido: 1790000559
- id_pedido: 1790001005

A ocorrência será registrada para validação
da regra de negócio.

## Insight de Qualidade

Durante a validação do campo status_pedido foram encontrados
2 registros com o valor "Aguardando".

A ocorrência indica possível evolução da regra de negócio
ou alteração na origem dos dados.

O registro foi identificado para análise sem remoção automática,
preservando a rastreabilidade da informação.

In [0]:
# ==================================================
# REGRA 5 - CONTROLE DE DUPLICIDADE DE ARQUIVOS
# ==================================================

arquivos_processados = (
    df_quality
    .select(
        "bronze_source_file"
    )
    .distinct()
)


total_arquivos = (
    arquivos_processados
    .count()
)


total_registros = (
    df_quality
    .count()
)


if total_arquivos == total_registros:

    print(
        "Controle de arquivos validado!"
    )

else:

    print(
        "Possível duplicidade encontrada!"
    )

Possível duplicidade encontrada!


In [0]:
# ==================================================
# ANÁLISE DOS ARQUIVOS DE ORIGEM
# ==================================================

arquivos_origem = (
    df_quality
    .select(
        "bronze_source_file"
    )
    .distinct()
)


display(
    arquivos_origem
)

bronze_source_file
real-time-data/2026/09/21/112245/ecommerce_pedidos.parquet
real-time-data/2026/09/21/112402/ecommerce_pedidos.parquet
real-time-data/2026/09/21/112602/ecommerce_pedidos.parquet
real-time-data/2026/09/21/112802/ecommerce_pedidos.parquet
real-time-data/2026/09/21/113002/ecommerce_pedidos.parquet
real-time-data/2026/09/21/113202/ecommerce_pedidos.parquet


In [0]:
# ==================================================
# CRIAÇÃO DO DATABASE METADATA
# ==================================================

spark.sql(
    """
    CREATE DATABASE IF NOT EXISTS grupo3_metadata
    """
)


print(
    "Database Metadata criado com sucesso!"
)

Database Metadata criado com sucesso!


In [0]:
# ==================================================
# CRIAÇÃO DO CONTROLE DE INGESTÃO
# ==================================================

from pyspark.sql.functions import lit, current_timestamp

controle_arquivos = (
    arquivos_origem
    .withColumn(
        "status",
        lit("PROCESSADO")
    )
    .withColumn(
        "processado_em",
        current_timestamp()
    )
)


(
    controle_arquivos
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "grupo3_metadata.ingestion_control"
    )
)


print(
    "Tabela de controle de ingestão criada com sucesso!"
)

Tabela de controle de ingestão criada com sucesso!


In [0]:
# ==================================================
# VISUALIZAÇÃO DO CONTROLE DE INGESTÃO
# ==================================================

display(
    spark.table(
        "grupo3_metadata.ingestion_control"
    )
)

bronze_source_file,status,processado_em
real-time-data/2026/09/21/112245/ecommerce_pedidos.parquet,PROCESSADO,2026-09-30T03:19:05.582Z
real-time-data/2026/09/21/112602/ecommerce_pedidos.parquet,PROCESSADO,2026-09-30T03:19:05.582Z
real-time-data/2026/09/21/112802/ecommerce_pedidos.parquet,PROCESSADO,2026-09-30T03:19:05.582Z
real-time-data/2026/09/21/113002/ecommerce_pedidos.parquet,PROCESSADO,2026-09-30T03:19:05.582Z
real-time-data/2026/09/21/113202/ecommerce_pedidos.parquet,PROCESSADO,2026-09-30T03:19:05.582Z
real-time-data/2026/09/21/112402/ecommerce_pedidos.parquet,PROCESSADO,2026-09-30T03:19:05.582Z


In [0]:
# ==================================================
# RELATÓRIO FINAL DE DATA QUALITY
# ==================================================

resultado_quality = [

    (
        "Schema obrigatório",
        "APROVADO",
        0
    ),

    (
        "Valor total > 0",
        "REPROVADO",
        quantidade_invalidos
    ),

    (
        "Datas válidas",
        "APROVADO",
        0
    ),

    (
        "Status permitido",
        "REPROVADO",
        quantidade_status_invalidos
    ),

    (
        "Controle de arquivos",
        "APROVADO",
        total_arquivos
    )
]


df_quality_resultado = spark.createDataFrame(
    resultado_quality,
    [
        "regra",
        "status",
        "quantidade"
    ]
)


display(
    df_quality_resultado
)

regra,status,quantidade
Schema obrigatório,APROVADO,0
Valor total > 0,REPROVADO,1
Datas válidas,APROVADO,0
Status permitido,REPROVADO,2
Controle de arquivos,APROVADO,6


In [0]:
# ==================================================
# SALVAR RESULTADO DATA QUALITY
# ==================================================

(
    df_quality_resultado
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "grupo3_metadata.data_quality_results"
    )
)


print(
    "Tabela de resultados Data Quality criada com sucesso!"
)

Tabela de resultados Data Quality criada com sucesso!


%md

# ✅ Conclusão - Data Quality de Pedidos

## Squad 2 - Real Time for Business

### Grupo 3 - Ecommerce Pedidos

---

# 🎯 Objetivo Alcançado

Este notebook implementou uma camada de validação de
qualidade dos dados de pedidos antes das próximas etapas
do pipeline.

Foram aplicadas regras técnicas e de negócio para garantir
maior confiabilidade, rastreabilidade e controle dos dados.

---

# 📥 Origem dos Dados

Os dados analisados foram carregados da camada Bronze:

```
Database:

grupo3_bronze


Tabela:

ecommerce_pedidos
```

A Bronze contém:

- Dados ingeridos do Real Time;
- Estrutura original dos registros;
- Informações de auditoria;
- Arquivos de origem.

---

# 🛡️ Regras de Qualidade Aplicadas

Foram implementadas validações para verificar:

---

## 1. Schema Obrigatório

Objetivo:

Garantir que os campos essenciais estejam presentes.

Campos validados:

```
id_pedido

id_cliente

dt_pedido

status_pedido

valor_total

metodo_pagamento
```

Resultado:

```
✅ APROVADO
```

---

## 2. Validação do Valor Total

Regra:

```
valor_total > 0
```

Resultado:

```
❌ REPROVADO

1 registro inconsistente encontrado
```

Detalhamento:

```
valor_total = -1.92
```

O registro foi identificado para análise,
preservando a rastreabilidade do dado.

---

## 3. Validação das Datas

Campos analisados:

```
dt_pedido

dt_previsao_entrega

dt_ultima_atualizacao_status
```

Resultado:

```
✅ APROVADO
```

Todos os campos temporais estavam válidos.

---

## 4. Validação de Status

Regra:

O campo `status_pedido` deve possuir apenas valores
permitidos pelo domínio definido.

Resultado:

```
❌ REPROVADO

2 registros encontrados
```

Valor identificado:

```
Aguardando
```

A ocorrência foi registrada para avaliação da regra
de negócio.

---

## 5. Controle de Arquivos Processados

Foi criada uma tabela de controle para evitar
reprocessamento de arquivos.

Tabela:

```
grupo3_metadata.ingestion_control
```

Resultado:

```
6 arquivos registrados

Status:

PROCESSADO
```

---

# 🗂️ Tabelas de Controle Criadas

## Controle de ingestão

```
grupo3_metadata.ingestion_control
```

Responsável por:

- Registrar arquivos processados;
- Evitar duplicidade de processamento;
- Garantir idempotência do pipeline.

---

## Resultado das validações

```
grupo3_metadata.data_quality_results
```

Responsável por:

- Armazenar resultados das regras;
- Registrar aprovações e falhas;
- Gerar evidências de qualidade.

---

# 📊 Resultado Geral

Resumo das validações:

| Regra | Resultado |
|---|---|
| Schema obrigatório | ✅ Aprovado |
| Valor total > 0 | ❌ 1 inconsistência |
| Datas válidas | ✅ Aprovado |
| Status permitido | ❌ 2 inconsistências |
| Controle de arquivos | ✅ Aprovado |

---

# 🏗️ Arquitetura Atualizada

O pipeline agora possui:

```
RAW

↓

🥉 Bronze

↓

🛡️ Data Quality

├── ingestion_control

└── data_quality_results

↓

🥈 Silver

↓

🥇 Gold
```

---

# 🛡️ Boas Práticas Aplicadas

Durante esta etapa foram aplicadas práticas de Engenharia
de Dados:

- Validação preventiva dos dados;
- Controle de qualidade;
- Rastreamento de inconsistências;
- Controle de reprocessamento;
- Organização de metadados;
- Persistência em Delta Lake.

---

# 🚀 Resultado Final

A camada Data Quality foi implementada com sucesso.

O pipeline agora possui mecanismos para identificar
problemas de estrutura, qualidade e consistência dos dados,
aumentando a confiabilidade das informações antes das
camadas analíticas.

Os dados estão preparados para a evolução futura da camada
Gold e criação de indicadores de negócio da Squad 2 - Grupo 3.